<a href="https://colab.research.google.com/github/potat0w/Aaagh-more-math/blob/main/textosql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

/bin/bash: line 1: nvidia-smi: command not found
cuda: False cpu


In [16]:
DRIVE_TEXT2SQL_ZIP = "/content/drive/MyDrive/text2sql_for_drive.zip"
DRIVE_SPIDER_ZIP = "/content/drive/MyDrive/spider_data.zip"

from pathlib import Path
import zipfile, os, shutil
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

TEXT2SQL_ZIP = Path(DRIVE_TEXT2SQL_ZIP)
SPIDER_ZIP = Path(DRIVE_SPIDER_ZIP)
assert TEXT2SQL_ZIP.is_file(), f"missing {TEXT2SQL_ZIP}"
assert SPIDER_ZIP.is_file(), f"missing {SPIDER_ZIP}"

WORK = Path("/content")
TEXT2SQL = WORK / "text2sql"

if TEXT2SQL.exists():
    shutil.rmtree(TEXT2SQL)
print("Unzipping", TEXT2SQL_ZIP, "->", WORK)
with zipfile.ZipFile(TEXT2SQL_ZIP) as z:
    z.extractall(WORK)

# zip is flat (src/, configs/, ...) not nested text2sql/
if not (TEXT2SQL / "src" / "common.py").is_file():
    TEXT2SQL.mkdir(parents=True, exist_ok=True)
    for name in ("src", "configs", "requirements.txt", "RUNBOOK.md", "colab_text2sql.ipynb", "assets", "logs"):
        src = WORK / name
        if src.exists() and not (TEXT2SQL / name).exists():
            shutil.move(str(src), str(TEXT2SQL / name))

assert (TEXT2SQL / "src" / "phase2_prep.py").is_file(), "re-upload text2sql_for_drive.zip to Drive"
os.chdir(TEXT2SQL)
print("src files:", sorted(p.name for p in (TEXT2SQL / "src").glob("*.py")))
print("cwd:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipping /content/drive/MyDrive/text2sql_for_drive.zip -> /content
src files: ['common.py', 'compare.py', 'eval.py', 'phase1_baseline_all.py', 'phase2_prep.py', 'phase3_train.py']
cwd: /content/text2sql


In [17]:
%pip install -q -r requirements.txt

In [18]:
import zipfile
from pathlib import Path

spider_root = WORK / "spider_data"
if not (spider_root / "train_spider.json").is_file():
    print("Unzipping", SPIDER_ZIP, "->", WORK)
    with zipfile.ZipFile(SPIDER_ZIP) as z:
        z.extractall(WORK)
    if not (spider_root / "train_spider.json").is_file():
        hits = list(WORK.glob("**/train_spider.json"))
        assert hits, "spider_data.zip must contain train_spider.json"
        spider_root = hits[0].parent

assert (spider_root / "train_spider.json").is_file(), spider_root

Path("results").mkdir(exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)

!python src/phase2_prep.py --spider_root {spider_root}

{
 "train_kept": 8656,
 "test_kept": 1032,
 "spider_root": "/content/spider_data",
 "out": [
  "data/train.jsonl",
  "data/test.jsonl",
  "results/phase2_data_stats.json"
 ]
}


In [21]:
import os
from google.colab import userdata

try:
    # Retrieve the token from Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("HF_TOKEN successfully set!")
except Exception as e:
    print(f"Could not find HF_TOKEN in Secrets. Please add it in the left sidebar (🔑 icon). Error: {e}")

HF_TOKEN successfully set!


In [23]:
!python src/eval.py --model Qwen/Qwen2.5-Coder-1.5B-Instruct --tag smoke --limit 16

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:07<00:00, 44.70it/s]
  0% 0/2 [00:00<?, ?it/s]^C


In [ ]:
!python src/eval.py --model Qwen/Qwen2.5-1.5B-Instruct --adapter /content/drive/MyDrive/text2sql_ckpt/qlora_Qwen2.5-1.5B-Instruct --tag phase4_Qwen2.5-1.5B-qlora

config.json: 100% 660/660 [00:00<00:00, 2.21MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 1.43MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 14.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 14.2MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 18.6MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 62.0M/3.09G [00:01<00:32, 94.5MB/s, 4.68MB/s  ]
model.safetensors: reconstructing file:   0% 15.0M/3.09G [00:01<03:23, 15.1MB/s, 4.29kB/s  ]
model.safetensors: downloading bytes:  32% 980M/3.09G [00:05<00:06, 302MB/s, 74.4MB/s  ]
model.safetensors: reconstructing file:   4% 135M/3.09G [00:08<03:22, 14.6MB/s, 4.46MB/s  ] 
model.safetensors: reconstructing file:   5% 164M/3.09G [00:08<02:28, 19.7MB/s, 8.94MB/s  ]
model.safetensors: downloading bytes:  33% 1.01G/3.09G [00:08<01:01, 33.6MB/s, 59.9MB/s  ]
model.safetensors: downloading bytes: 